# Classer des intentions avec un LLM, et comparer

Hier, les modèles classiques, les *embeddings* et un encodeur affiné ont classé les demandes de MASSIVE. Aujourd'hui, un LLM, sans entraînement :

1. Sans exemple : la liste des intentions décrites, une sortie contrainte
2. Avec quelques exemples dans le prompt
3. Comparer toutes les méthodes
4. Lire les erreurs

Ce notebook va plus vite avec un GPU : *Exécution > Modifier le type d'exécution > GPU T4*. Sans GPU, un LLM plus petit (Qwen3-1.7B au lieu de Qwen3-4B) tourne sur le processeur, plus lentement : il ne classe alors que 10 demandes au lieu de 300 (sept minutes environ pour les deux prompts). De quoi écrire et faire tourner le code, pas de quoi conclure : sur 10 demandes, et avec un autre modèle, votre F1 de l'exercice 2 ne dit pas si les exemples aident. L'exercice 3 charge alors les résultats de Qwen3-4B sur les 300 demandes, calculés sur GPU par le même code : le tableau, et la réponse à l'exercice 2, sont ceux du GPU. L'exercice 4, lui, porte sur tout le test, avec ou sans GPU.

## Données et modèle

Les résultats d'hier : chargez vos fichiers `predictions-fr.csv` et `timings-fr.csv` (panneau *Fichiers* à gauche), ou utilisez les nôtres, téléchargés par la cellule suivante s'ils manquent. Ollama s'installe et télécharge le modèle en arrière-plan.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh massive-intents
!test -f predictions-fr.csv || cp massive-intents/lab-results/predictions-fr.csv .
!test -f timings-fr.csv || cp massive-intents/lab-results/timings-fr.csv .
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

import os
import subprocess

import torch

GPU = torch.cuda.is_available()
# Sur GPU, Qwen3-4B. Sur CPU, Qwen3-1.7B, deux fois plus rapide : un modèle « hybride »,
# qui réfléchit avant de répondre, sauf avec reasoning_effort="none" (dans les appels).
MODELE = "qwen3:4b-instruct-2507-q4_K_M" if GPU else "qwen3:1.7b"
# Sur GPU, 8 demandes à la fois ; sur CPU, une : le début commun des prompts (les
# consignes) n'est alors calculé qu'une fois.
paralleles = "8" if GPU else "1"
env = {**os.environ, "OLLAMA_NUM_PARALLEL": paralleles, "OLLAMA_KEEP_ALIVE": "-1"}
subprocess.Popen(
  ["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
telechargement = subprocess.Popen(
  f"sleep 5 && ollama pull {MODELE}",
  shell=True,
  stdout=subprocess.DEVNULL,
  stderr=subprocess.DEVNULL,
)

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor
from typing import Literal

import pandas as pd
from openai import OpenAI
from pydantic import create_model
from sklearn.metrics import f1_score

train = pd.read_parquet("massive-intents/fr-train.parquet")
intents = pd.read_csv("massive-intents/intents.csv")
test = pd.read_parquet("massive-intents/fr-test.parquet")
# Les textes des demandes, d'après leur id.
predictions = pd.read_csv("predictions-fr.csv", dtype={"id": str}).merge(
  test[["id", "text"]], on="id"
)
durees = pd.read_csv("timings-fr.csv")
# 300 demandes de test, tirées au hasard : un LLM est trop lent pour les 2 974. Sans GPU,
# 10 : chaque demande y prend plusieurs secondes.
echantillon = predictions.sample(300 if GPU else 10, random_state=0).reset_index(
  drop=True
)
# Le F1 macro de chaque méthode porte sur les intentions d'`echantillon` : sinon, une
# prédiction d'une intention absente lui ajouterait une classe à F1 nul.
presentes = sorted(echantillon.label.unique())
intentions = intents.label.tolist()
print(len(intentions), "intentions")

assert telechargement.wait() == 0, "Téléchargement du modèle raté"
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# Un premier appel charge le modèle (sur le GPU s'il y en a un), où il reste : avant les calculs en parallèle.
client.chat.completions.create(
  model=MODELE, messages=[{"role": "user", "content": "Bonjour"}], max_tokens=1
)
print(MODELE, "prêt")

## 1. Sans exemple

Le schéma de sortie, `Intention`, n'a qu'un champ, `intention`, dont la valeur est forcément l'une des 60 étiquettes : le modèle ne peut pas répondre hors liste.

Écrivez le prompt système (`SANS_EXEMPLE`) : la tâche, puis la liste des intentions avec leur description (`intents.description_fr`). Écrivez `classe(texte, systeme)` qui renvoie l'intention choisie, classez les demandes d'`echantillon` (300 ; sans GPU, 10) en parallèle (avec une fonction `classe_tout(systeme)`, que la suite du TP réutilise : rangez ses résultats dans `sans_exemple` et `secondes_sans`), et mesurez le F1 macro et le temps par texte.

**Documentation utile**

- [`pydantic.create_model`](https://docs.pydantic.dev/latest/concepts/models/#dynamic-model-creation)
- [`DataFrame.itertuples`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.itertuples.html)
- [`ThreadPoolExecutor.map`](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.Executor.map)
- [`sklearn.metrics.f1_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)

In [ ]:
Intention = create_model("Intention", intention=(Literal[tuple(intentions)], ...))

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `intents.itertuples()` : `ligne.label` et `ligne.description_fr` ; une ligne `- étiquette : description` par intention, jointes par `"\n"`
- le prompt : la tâche (« tu classes les demandes faites à un assistant vocal selon leur intention »), puis « choisis exactement une intention de cette liste » et la liste
- `client.chat.completions.parse(model=MODELE, messages=[{"role": "system", "content": systeme}, {"role": "user", "content": texte}], response_format=Intention, temperature=0, reasoning_effort="none")` ; l'intention est dans `reponse.choices[0].message.parsed.intention`
- `reasoning_effort="none"` : le modèle répond sans phase de réflexion (sans GPU, Qwen3-1.7B en aurait une, qui multiplie le temps de réponse)
- `ThreadPoolExecutor(8)` : 8 appels en parallèle ; `pool.map(fonction, echantillon.text)` renvoie les résultats dans l'ordre des textes
- `f1_score(echantillon.label, prédictions, average="macro", labels=presentes)` : `average="macro"` donne le même poids à chaque classe, `labels` limite la moyenne aux intentions d'`echantillon` ; le temps par texte en ms : `1000 * secondes / len(echantillon)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
LISTE = "\n".join(f"- {ligne.label} : {ligne.description_fr}" for ligne in ...)
SANS_EXEMPLE = f"""Tu classes les demandes faites à un assistant vocal selon leur intention.
Choisis exactement une intention de cette liste :
{LISTE}"""


def classe(texte: str, systeme: str) -> str:
  reponse = client.chat.completions.parse(
    model=MODELE,
    messages=[
      {"role": "system", "content": systeme},
      {"role": "user", "content": texte},
    ],
    response_format=...,
    temperature=0,
    reasoning_effort="none",
  )
  return reponse.choices[0].message.parsed.intention


def classe_tout(systeme: str) -> tuple[list[str], float]:
  debut = time.time()
  with ThreadPoolExecutor(8) as pool:
    trouvees = list(
      pool.map(lambda texte: classe(texte, systeme), ...)
    )  # les textes d'`echantillon`
  return trouvees, time.time() - debut


sans_exemple, secondes_sans = classe_tout(SANS_EXEMPLE)
print(
  f"F1 macro {f1_score(echantillon.label, ..., average=..., labels=presentes):.3f}, {1000 * secondes_sans / len(echantillon):.0f} ms/texte"
)
```

### Solution

In [ ]:
LISTE = "\n".join(
  f"- {ligne.label} : {ligne.description_fr}" for ligne in intents.itertuples()
)
SANS_EXEMPLE = f"""Tu classes les demandes faites à un assistant vocal selon leur intention.
Choisis exactement une intention de cette liste :
{LISTE}"""


def classe(texte: str, systeme: str) -> str:
  reponse = client.chat.completions.parse(
    model=MODELE,
    messages=[
      {"role": "system", "content": systeme},
      {"role": "user", "content": texte},
    ],
    response_format=Intention,
    temperature=0,
    reasoning_effort="none",
  )
  return reponse.choices[0].message.parsed.intention


def classe_tout(systeme: str) -> tuple[list[str], float]:
  debut = time.time()
  with ThreadPoolExecutor(8) as pool:
    trouvees = list(pool.map(lambda texte: classe(texte, systeme), echantillon.text))
  return trouvees, time.time() - debut


sans_exemple, secondes_sans = classe_tout(SANS_EXEMPLE)
print(
  f"F1 macro {f1_score(echantillon.label, sans_exemple, average='macro', labels=presentes):.3f}, {1000 * secondes_sans / len(echantillon):.0f} ms/texte"
)

# Sur GPU (Qwen3-4B, 300 demandes, une RTX A4000) : F1 macro 0,721, 256 ms par texte.
# Sans GPU, sur 10 demandes et avec Qwen3-1.7B, ce F1 ne se compare pas.

## 2. Avec quelques exemples

Ajoutez au prompt deux exemples d'entraînement par intention. Le F1 progresse-t-il ? Combien de tokens le prompt compte-t-il maintenant, contre combien sans exemple, et qu'est-ce que cela coûterait par une API, à 0,10 € par million de tokens d'entrée, pour 1 million de demandes ?

Sans GPU, votre F1 ne répond pas à la première question : sur 10 demandes, une erreur de plus ou de moins le change d'une dizaine de points, et Qwen3-1.7B n'est pas le modèle qui tourne sur GPU. La réponse est dans le tableau de l'exercice 3, dont la première cellule charge les résultats du GPU sur les 300 demandes. Les tokens et le coût, eux, ne dépendent que du prompt.

**Documentation utile**

- [`DataFrameGroupBy.sample`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.sample.html)
- [Ollama : compatibilité OpenAI](https://docs.ollama.com/api/openai-compatibility)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `train.groupby("label").sample(2, random_state=0)` : 2 exemples par intention, toujours les mêmes
- un exemple par ligne, à la suite du prompt de l'exercice précédent : `« texte » -> intention`
- `classe_tout(AVEC_EXEMPLES)` : la fonction de l'exercice précédent, avec le nouveau prompt
- `reponse.usage.prompt_tokens` : le nombre de tokens du prompt, dans la réponse d'un appel à `client.chat.completions.create(..., max_tokens=1)` (un seul token généré suffit), pour chacun des deux prompts

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
exemples = train.groupby("label").sample(2, random_state=0)
EXEMPLES = "\n".join(f"« {ligne.text} » -> {ligne.label}" for ligne in ...)
AVEC_EXEMPLES = f"{SANS_EXEMPLE}\n\nExemples :\n{EXEMPLES}"

avec_exemples, secondes_avec = classe_tout(...)
print(
  f"F1 macro {f1_score(echantillon.label, avec_exemples, average='macro', labels=presentes):.3f}, {1000 * secondes_avec / len(echantillon):.0f} ms/texte"
)

for methode, systeme in (
  ("LLM sans exemple", SANS_EXEMPLE),
  ("LLM avec exemples", AVEC_EXEMPLES),
):
  usage = client.chat.completions.create(
    model=MODELE,
    messages=[
      {"role": "system", "content": ...},
      {"role": "user", "content": echantillon.text[0]},
    ],
    max_tokens=1,
  ).usage
  # 1 million de demandes de prompt_tokens tokens, à 0,10 € par million de tokens :
  print(
    f"{methode} : {usage.prompt_tokens} tokens de prompt,",
    f"{... * 0.10:.0f} € pour 1 million de demandes",
  )
```

### Solution

In [ ]:
exemples = train.groupby("label").sample(2, random_state=0)
EXEMPLES = "\n".join(
  f"« {ligne.text} » -> {ligne.label}" for ligne in exemples.itertuples()
)
AVEC_EXEMPLES = f"{SANS_EXEMPLE}\n\nExemples :\n{EXEMPLES}"

avec_exemples, secondes_avec = classe_tout(AVEC_EXEMPLES)
print(
  f"F1 macro {f1_score(echantillon.label, avec_exemples, average='macro', labels=presentes):.3f}, {1000 * secondes_avec / len(echantillon):.0f} ms/texte"
)

for methode, systeme in (
  ("LLM sans exemple", SANS_EXEMPLE),
  ("LLM avec exemples", AVEC_EXEMPLES),
):
  usage = client.chat.completions.create(
    model=MODELE,
    messages=[
      {"role": "system", "content": systeme},
      {"role": "user", "content": echantillon.text[0]},
    ],
    max_tokens=1,
  ).usage
  # 1 million de demandes de prompt_tokens tokens, à 0,10 € par million de tokens :
  print(
    f"{methode} : {usage.prompt_tokens} tokens de prompt,",
    f"{usage.prompt_tokens * 0.10:.0f} € pour 1 million de demandes",
  )

# Un long préfixe commun (consignes + exemples) est exactement ce que le cache de prompt
# des API facture à prix réduit.
# Sur GPU (Qwen3-4B, 300 demandes) : F1 macro 0,744 avec exemples contre 0,721 sans,
# 359 ms par texte contre 256, et 3 042 tokens de prompt contre 954 (304 € contre 95
# pour 1 million de demandes). Deux points de F1 de plus, sur 300 demandes, ne suffisent
# pas à établir que les exemples aident, pour trois fois plus de tokens.

## 3. Comparer toutes les méthodes

Rassemblez dans un tableau, sur les **mêmes** demandes d'`echantillon` : le F1 macro de chaque méthode d'hier (colonnes d'`echantillon`) et des deux du jour, leur temps d'entraînement et leur temps par texte (`durees`). Laquelle mettriez-vous en production pour 10 millions de demandes par mois ? Et pour un prototype à livrer demain ?

Trois précautions avant de lire le tableau :

- **Les temps ne viennent pas du même matériel.** Les méthodes sur TF-IDF ont tourné sur processeur, les *embeddings* et l'encodeur sur le GPU d'hier (le vôtre, ou celui de nos résultats publiés), le LLM sur le GPU de ce notebook (sans GPU, celui de nos résultats publiés). À matériel égal, sur un processeur à 2 cœurs, l'encodeur affiné prend plus d'une centaine de ms par texte.
- **Le F1 n'est pas celui d'hier.** Il porte sur 300 demandes au lieu de 2 974, et sur les seules intentions qu'elles contiennent : avec nos résultats, 0,862 pour l'encodeur, contre 0,835 hier. D'un tirage de 300 demandes à l'autre, il bouge d'environ ± 6 points : deux méthodes à quelques points d'écart ne se départagent pas.
- **Sans GPU**, la cellule suivante remplace les résultats de vos 10 demandes par ceux de Qwen3-4B sur les 300, calculés sur GPU par le même code : `echantillon`, `sans_exemple`, `avec_exemples` et leurs temps. Le tableau porte alors sur les 300 demandes, comme sur GPU.

**Documentation utile**

- [`sklearn.metrics.f1_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)
- [`DataFrame.set_index`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.set_index.html)

In [ ]:
# Sans GPU, les exercices 1 et 2 n'ont classé que 10 demandes, avec Qwen3-1.7B : on charge
# les prédictions de Qwen3-4B sur les 300, calculées sur GPU par le même code, et leurs
# temps.
if not GPU:
  echantillon = predictions.sample(300, random_state=0).reset_index(drop=True)
  presentes = sorted(echantillon.label.unique())
  llm = pd.read_csv(
    "massive-intents/lab-results/llm-predictions-fr.csv", dtype={"id": str}
  )
  assert llm.id.tolist() == echantillon.id.tolist(), "Pas les demandes d'`echantillon`"
  sans_exemple, avec_exemples = llm.zero_shot.tolist(), llm.few_shot.tolist()
  durees_llm = pd.read_csv("massive-intents/lab-results/llm-timings-fr.csv")
  ms_par_texte = durees_llm.set_index("method").ms_per_text
  secondes_sans = ms_par_texte["LLM sans exemple"] * len(echantillon) / 1000
  secondes_avec = ms_par_texte["LLM avec exemples"] * len(echantillon) / 1000
  print(
    "Résultats du GPU chargés :", len(echantillon), "demandes, qwen3:4b-instruct-2507"
  )

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- les méthodes d'hier sont les colonnes d'`echantillon` autres que `id`, `label` et `text`
- `durees.set_index("method").loc[méthode]` : la ligne de temps d'une méthode (`.train_s`, `.ms_per_text`)
- `f1_score(echantillon.label, echantillon[méthode], average="macro", labels=presentes)` : `average="macro"` donne le même poids à chaque classe, `labels` limite la moyenne aux intentions d'`echantillon`
- les deux méthodes du jour : `sans_exemple` et `avec_exemples`, sans entraînement (0 s), et leur temps par texte `1000 * secondes / len(echantillon)`
- `pd.DataFrame(lignes, columns=[...]).round(3)` à partir d'une liste de tuples

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
methodes = [
  colonne for colonne in echantillon.columns if colonne not in ("id", "label", "text")
]
lignes = []
for methode in methodes:
  duree = durees.set_index("method").loc[...]
  lignes.append(
    (
      methode,
      f1_score(echantillon.label, echantillon[...], average="macro", labels=presentes),
      duree.train_s,
      duree.ms_per_text,
    )
  )
lignes.append(
  (
    "LLM sans exemple",
    f1_score(echantillon.label, ..., average="macro", labels=presentes),
    0,
    1000 * secondes_sans / len(echantillon),
  )
)
lignes.append(
  (
    "LLM avec exemples",
    f1_score(echantillon.label, ..., average="macro", labels=presentes),
    0,
    1000 * secondes_avec / len(echantillon),
  )
)
pd.DataFrame(
  lignes, columns=["méthode", "F1 macro", "entraînement (s)", "ms/texte"]
).round(3)
```

### Solution

In [ ]:
methodes = [
  colonne for colonne in echantillon.columns if colonne not in ("id", "label", "text")
]
lignes = []
for methode in methodes:
  duree = durees.set_index("method").loc[methode]
  lignes.append(
    (
      methode,
      f1_score(
        echantillon.label, echantillon[methode], average="macro", labels=presentes
      ),
      duree.train_s,
      duree.ms_per_text,
    )
  )
lignes.append(
  (
    "LLM sans exemple",
    f1_score(echantillon.label, sans_exemple, average="macro", labels=presentes),
    0,
    1000 * secondes_sans / len(echantillon),
  )
)
lignes.append(
  (
    "LLM avec exemples",
    f1_score(echantillon.label, avec_exemples, average="macro", labels=presentes),
    0,
    1000 * secondes_avec / len(echantillon),
  )
)
pd.DataFrame(
  lignes, columns=["méthode", "F1 macro", "entraînement (s)", "ms/texte"]
).round(3)

# Les temps ne se comparent qu'à matériel égal : sur GPU, l'encodeur affiné prend moins
# d'une ms par texte ; sur processeur, plus d'une centaine, et le LLM plusieurs secondes.
# En production à gros volume : l'encodeur affiné, ou les embeddings, à peine moins
# justes sur tout le test (0,810 contre 0,835 hier ; ici, 0,804 contre 0,862 : sur 300
# demandes, l'écart varie d'un tirage à l'autre). Le LLM (Qwen3-4B sur GPU : 0,721 sans
# exemple, 0,744 avec) reste loin derrière eux, parmi les méthodes sur TF-IDF (de 0,691 à
# 0,776), que 300 demandes ne départagent pas, pour 256 et 359 ms par texte : des
# centaines de fois plus lent que l'encodeur. Pour un prototype sans données étiquetées :
# le LLM, tout de suite.

## 4. Lire les erreurs

La méthode de la diapositive « Lire les erreurs », sur les erreurs de l'encodeur affiné parmi les 2 974 demandes de test. Ses logits sont publiés (`massive-intents/lab-results/encoder-logits-fr.npy`, déjà téléchargé : une ligne par demande de `test`, une colonne par intention, dans l'ordre alphabétique). Ce sont ceux de notre encodeur, entraîné sur GPU, et non du vôtre, dont le TP d'hier n'a gardé que les prédictions : l'exercice est le même avec ou sans GPU.

Transformez les logits en probabilités (`softmax`), gardez les erreurs et triez-les par confiance décroissante. Lisez les 10 premières, avec la description des intentions (`intents`) : erreur du modèle, ou étiquette fausse ? Regroupez ensuite les erreurs par paire (intention attendue, intention prédite) : quelles paires reviennent le plus souvent, et qu'en feriez-vous ?

**Documentation utile**

- [`numpy.load`](https://numpy.org/doc/stable/reference/generated/numpy.load.html)
- [`scipy.special.softmax`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.softmax.html)
- [`DataFrame.assign`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.assign.html)
- [`DataFrame.sort_values`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html)
- [`DataFrame.value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.value_counts.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `np.load(chemin)` : les logits, 2 974 lignes (dans l'ordre de `test`) et 60 colonnes ; `np.array(sorted(train.label.unique()))` : les intentions, dans l'ordre des colonnes
- `softmax(logits, axis=1)` : une distribution de probabilités par demande
- `proba.argmax(axis=1)` : le numéro de l'intention prédite, que `classes[...]` convertit en intention ; `proba.max(axis=1)` : sa probabilité, la confiance du modèle
- `test.assign(predite=..., confiance=...)` : `test`, avec deux colonnes de plus
- `encodeur[encodeur.predite != encodeur.label]` : les erreurs (vous devez en trouver 371), puis `.sort_values("confiance", ascending=False)` : les plus sûres d'abord
- `erreurs.value_counts(["label", "predite"])` : le nombre d'erreurs de chaque paire, de la plus fréquente à la plus rare
- `intents.set_index("label").description_fr` : la description de chaque intention, pour trancher entre le modèle et l'étiquette
- `pd.set_option("display.max_colwidth", None)` : pandas affiche les textes en entier ; `display(tableau)` affiche un tableau au milieu d'une cellule

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
import numpy as np
from IPython.display import display
from scipy.special import softmax

pd.set_option("display.max_colwidth", None)  # les textes en entier
logits = np.load("massive-intents/lab-results/encoder-logits-fr.npy")
classes = np.array(sorted(train.label.unique()))  # l'ordre des colonnes des logits
proba = softmax(..., axis=1)
encodeur = test.assign(predite=classes[proba.argmax(axis=1)], confiance=...)
erreurs = encodeur[encodeur.predite != ...].sort_values("confiance", ascending=...)
print(len(erreurs), "erreurs,", (erreurs.confiance > 0.99).sum(), "à plus de 0,99")
# les colonnes : le texte, l'étiquette, la prédiction et la confiance
display(erreurs[[..., ..., ..., ...]].head(10))
erreurs.value_counts([..., ...]).head(10)
```

### Solution

In [ ]:
import numpy as np
from IPython.display import display
from scipy.special import softmax

pd.set_option("display.max_colwidth", None)  # les textes en entier
logits = np.load("massive-intents/lab-results/encoder-logits-fr.npy")
classes = np.array(sorted(train.label.unique()))  # l'ordre des colonnes des logits
proba = softmax(logits, axis=1)
encodeur = test.assign(
  predite=classes[proba.argmax(axis=1)], confiance=proba.max(axis=1)
)
erreurs = encodeur[encodeur.predite != encodeur.label].sort_values(
  "confiance", ascending=False
)
print(
  len(erreurs),
  "erreurs,",
  (erreurs.confiance > 0.99).sum(),
  "à plus de 0,99 de confiance",
)
display(erreurs[["text", "label", "predite", "confiance"]].head(10))
erreurs.value_counts(["label", "predite"]).head(10)

# 371 erreurs, dont 84 à plus de 0,99 de confiance. Parmi les 10 plus sûres, presque
# toutes sont des étiquettes fausses ou discutables, comme la diapositive l'annonce :
# quatre demandes d'ajout d'une adresse à un contact (« vas dans mes contacts et ajoute
# une nouvelle adresse mail ») sont étiquetées email_sendemail, alors que
# email_addcontact est « Ajouter un contact ou une adresse e-mail » ; le cours d'une
# action est étiqueté news_query au lieu de qa_stock, l'état du trafic transport_query
# au lieu de transport_traffic. « mélange les chansons par coldplay » se discute
# (music_settings : « aléatoire ou répétition »). Les paires les plus fréquentes,
# qa_factoid et general_quirky (14 erreurs dans chaque sens), puis general_quirky prise
# pour news_query (10) : des intentions mal délimitées, general_quirky étant « hors des
# autres catégories ». Mieux les définir, ou les fusionner, rapporte plus qu'un modèle
# plus gros.